# Generative LLM — `career_position` Annotation

**Model:** Gemini 2.5 Flash (Google)  
**Granularity:** Fine-grained codes  
**Prompting:** zero-shot (set `N_SHOTS > 0` for few-shot)  

> Requires `GEMINI_API_KEY` in your `.env` file.

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import re
import time
from tqdm.auto import tqdm
from openai import OpenAI  # Gemini supports OpenAI-compatible API

from corex_eval import evaluate, load_inputs, load_training_data, career_position_to_sector

# --- Config ---
MODEL_ID = "gemini-2.5-flash"
N_SHOTS  = 0    # 0 = zero-shot; set to 3 or 5 for few-shot
SEED     = 42

## 2. Set up client

In [ ]:
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ["GEMINI_API_KEY"],
)

## 3. Load valid codes

In [ ]:
train_df = load_training_data(
    task="annotation",
    variable="career_position",
    features=["job_description_label", "workplace_label", "career_position"],
)
train_df = train_df.dropna(subset=["job_description_label", "career_position"]).reset_index(drop=True)
valid_codes = sorted(train_df["career_position"].unique())
print(f"{len(valid_codes)} valid career position codes")

## 4. Build prompt & prediction parser

In [ ]:
def build_system_prompt(valid_codes):
    codes_str = "\n".join(f"  {c}" for c in valid_codes)
    return (
        "You are an expert political scientist coding political career histories "
        "using the CoREx codebook.\n\n"
        "Task: given a job description, assign the single most appropriate career position code.\n"
        "Respond with ONLY the exact code label as listed below — "
        'for example: "105 = Minister with portfolio (including European Commissioner)"\n'
        "Do not add any explanation, commentary, or extra formatting.\n\n"
        f"Valid career position codes:\n{codes_str}"
    )

def build_messages(job_description, workplace, system_prompt, shot_examples):
    user_content = f"Job title: {job_description}\nWorkplace: {workplace or 'unknown'}"
    messages = [{"role": "system", "content": system_prompt}]
    for desc, wp, code in shot_examples:
        messages.append({"role": "user", "content": f"Job title: {desc}\nWorkplace: {wp or 'unknown'}"})
        messages.append({"role": "assistant", "content": code})
    messages.append({"role": "user", "content": user_content})
    return messages

def parse_prediction(response_text, valid_codes):
    """Robustly map model response to the nearest valid code."""
    text = response_text.strip().strip("\"'")
    # 1. Exact match
    if text in valid_codes:
        return text
    # 2. Match by code number (e.g. "105" → "105 = Minister with portfolio")
    code_map = {c.split(" = ", 1)[0].strip(): c for c in valid_codes if " = " in c}
    for code_num, full_code in code_map.items():
        if re.search(r"\b" + re.escape(code_num) + r"\b", text):
            return full_code
    # 3. Case-insensitive description substring match
    text_lower = text.lower()
    for code in valid_codes:
        desc = code.split(" = ", 1)[1].lower() if " = " in code else code.lower()
        if desc in text_lower or text_lower in desc:
            return code
    # 4. No match — will be scored wrong
    return text

SYSTEM_PROMPT = build_system_prompt(valid_codes)
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

## 5. Few-shot examples (skipped when `N_SHOTS=0`)

In [ ]:
if N_SHOTS > 0:
    shot_pool = (
        train_df.assign(sector=train_df["career_position"].map(career_position_to_sector))
        .groupby("sector", group_keys=False)
        .apply(lambda g: g.sample(1, random_state=SEED))
        .reset_index(drop=True)
    )
    shot_examples = (
        shot_pool.sample(min(N_SHOTS, len(shot_pool)), random_state=SEED)
        [["job_description_label", "workplace_label", "career_position"]]
        .values.tolist()
    )
    print(f"Using {len(shot_examples)} few-shot examples")
else:
    shot_examples = []
    print("Zero-shot mode")

## 6. Load test inputs

In [ ]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")
test_df.head()

## 7. Run inference

In [ ]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    messages = build_messages(
        row["job_description_label"],
        row.get("workplace_label", ""),
        SYSTEM_PROMPT,
        shot_examples,
    )
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=0,
            max_tokens=64,
            seed=SEED,
        )
        raw = response.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
        time.sleep(0.1)
    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

## 8. Evaluate

In [ ]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
    granularity="fine",
)